# Hurtownia danych – Online Retail (wersja z podpowiedziami)

## 0. Import bibliotek

👉 Wskazówka: użyj Pandas


In [2]:
import pandas as pd
import numpy as np


## 1. Wczytanie danych

👉 Wskazówka:
- użyj `pd.read_csv`
- sprawdź `head()` i `info()`


In [4]:
# Wczytanie danych
df = pd.read_csv("Online_Retail.csv", encoding='latin1')

print("Przykładowe dane:")
display(df.head())

print("\nInformacje o zbiorze:")
df.info()

Przykładowe dane:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/10 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/10 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/10 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/10 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/10 8:26,3.39,17850.0,United Kingdom



Informacje o zbiorze:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 453489 entries, 0 to 453488
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    453489 non-null  object 
 1   StockCode    453489 non-null  object 
 2   Description  452115 non-null  object 
 3   Quantity     453489 non-null  int64  
 4   InvoiceDate  453489 non-null  object 
 5   UnitPrice    453489 non-null  float64
 6   CustomerID   341144 non-null  float64
 7   Country      453488 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 27.7+ MB


## 2. Czyszczenie danych

👉 Wskazówki:
- usuń wiersze bez CustomerID → `dropna`
- usuń anulacje → InvoiceNo zaczyna się od 'C'
- usuń Quantity <= 0
- usuń UnitPrice <= 0

👉 Podpowiedź: filtruj dane krok po kroku


## Zgodnie z wymogami, usuwamy braki, anulacje, ujemne wartości i dodajemy przychód

In [5]:
# 1. Usunięcie wierszy bez CustomerID
df = df.dropna(subset=["CustomerID"])

# 2. Usunięcie anulowanych transakcji (InvoiceNo zaczyna się od 'C')
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]

# 3. Usunięcie Quantity <= 0 oraz UnitPrice <= 0
df = df[df["Quantity"] > 0]
df = df[df["UnitPrice"] > 0]

# 4. Konwersja daty do odpowiedniego formatu
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

# 5. Usunięcie duplikatów
df = df.drop_duplicates()

# 6. Dodanie miary Revenue (Przychód)
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

print(f"Liczba wierszy po czyszczeniu: {len(df)}")


/tmp/ipykernel_7667/2042773388.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])


Liczba wierszy po czyszczeniu: 329351


## 3. Wybór grain

👉 Pomyśl:
- czy jeden wiersz to produkt?
- czy faktura?

👉 Najczęstsza odpowiedź: linia faktury (InvoiceNo + StockCode)

**Opisz swoją decyzję (2-3 zdania)**


Zdecydowałem się na ziarno na poziomie pojedynczej pozycji na fakturze (kombinacja InvoiceNo + StockCode). Wybrałem ten poziom szczegółowości, ponieważ zapewnia mi on największą elastyczność i umożliwia precyzyjną analizę sprzedaży poszczególnych produktów. Konsekwencją teo wyboru jest większy rozmiar przechowywanych danych, ale dzięki temu można swobodnie agregować wyniki do wyższych poziomów – np. do całych zamówień, poszczególnych klientów czy przedziałów czasowych.

## 4. Schemat gwiazdy

👉 Minimum:
- FactSales
- DimCustomer
- DimProduct
- DimDate

👉 Pomyśl:
- jakie kolumny będą w każdej tabeli?


Zaprojektowałem schemat gwiazdy składający się z centralnej tabeli faktów FactSales oraz trzech wymiarów: DimCustomer, DimProduct i DimDate. W wymiarach przechowywane są atrybuty opisowe oraz wygenerowane klucze sztuczne. Z kolei tabela faktów zawiera miary biznesowe (Quantity, Revenue) i do relacji z wymiarami wykorzystuje wyłącznie wspomniane klucze sztuczne. Taki zabieg pozwala uniezależnić strukturę hurtowni od ewentualnych zmian kluczy naturalnych w systemach źródłowych.

## 5. Budowa wymiarów

👉 Wskazówki:
- DimCustomer → CustomerID, Country
- DimProduct → StockCode, Description
- DimDate → data z InvoiceDate

👉 Pamiętaj o kluczach sztucznych (ID)


In [6]:
# 1. DimProduct
# Pobieramy unikalne produkty. Bierzemy pierwszy opis dla danego kodu, aby uniknąć duplikatów
dim_product = df.groupby("StockCode")["Description"].first().reset_index()
dim_product["Product_SK"] = dim_product.index + 1 # Klucz sztuczny


# 2. DimDate
# Tworzymy unikalne daty na podstawie InvoiceDate
df["Date"] = df["InvoiceDate"].dt.date
dim_date = pd.DataFrame({"Date": df["Date"].unique()})
dim_date["Year"] = pd.to_datetime(dim_date["Date"]).dt.year
dim_date["Month"] = pd.to_datetime(dim_date["Date"]).dt.month
dim_date["Day"] = pd.to_datetime(dim_date["Date"]).dt.day
dim_date = dim_date.sort_values("Date").reset_index(drop=True)
dim_date["Date_SK"] = dim_date.index + 1 # Klucz sztuczny


# 3. DimCustomer (z implementacją SCD typu 2)
# Śledzimy zmianę przypisania klienta do kraju (Country) w czasie
dim_customer = df.groupby(["CustomerID", "Country"], as_index=False).agg(
    valid_from=("InvoiceDate", "min")
).sort_values(by=["CustomerID", "valid_from"]).reset_index(drop=True)

# Klucz sztuczny
dim_customer["Customer_SK"] = dim_customer.index + 1

# Ustawienie valid_to oraz current_flag dla SCD2
dim_customer["valid_to"] = dim_customer.groupby("CustomerID")["valid_from"].shift(-1)
# Jeśli nie ma nowszego rekordu, ustawiamy datę daleko w przyszłości
dim_customer["valid_to"] = dim_customer["valid_to"].fillna(pd.to_datetime("2099-12-31"))

dim_customer["current_flag"] = dim_customer["valid_to"] == pd.to_datetime("2099-12-31")
dim_customer["current_flag"] = dim_customer["current_flag"].map({True: "Y", False: "N"})

print("Przykładowy DimCustomer z SCD2:")
display(dim_customer.head())


Przykładowy DimCustomer z SCD2:


,CustomerID,Country,valid_from,Customer_SK,valid_to,current_flag
0,12346.0,United Kingdom,2011-01-18 10:01:00,1,2099-12-31,Y
1,12347.0,Iceland,2010-12-07 14:57:00,2,2099-12-31,Y
2,12348.0,Finland,2010-12-16 19:09:00,3,2099-12-31,Y
3,12350.0,Norway,2011-02-02 16:01:00,4,2099-12-31,Y
4,12352.0,Norway,2011-02-16 12:33:00,5,2099-12-31,Y


## 6. Budowa tabeli faktów

👉 Wskazówki:
- połącz dane z wymiarami
- dodaj miary:
  - Quantity
  - Revenue = Quantity * UnitPrice



In [7]:
# FactSales
# 1. Dołączamy Product_SK (łączymy po kluczu naturalnym StockCode)
fact_sales = df.merge(dim_product[["StockCode", "Product_SK"]], on="StockCode", how="left")

# 2. Dołączamy Date_SK (łączymy po kolumnie Date)
fact_sales = fact_sales.merge(dim_date[["Date", "Date_SK"]], on="Date", how="left")

# 3. Dołączamy Customer_SK
# Łączymy po CustomerID oraz Country, aby poprawnie złapać historyczną wersję klienta (zgodnie z SCD2)
fact_sales = fact_sales.merge(dim_customer[["CustomerID", "Country", "Customer_SK"]],
                              on=["CustomerID", "Country"], how="left")

# 4. Wybieramy docelowe kolumny: wyłącznie klucze sztuczne z wymiarów, nr faktury i miary
fact_sales = fact_sales[[
    "Customer_SK", "Product_SK", "Date_SK", "InvoiceNo", "Quantity", "Revenue"
]]

print("Tabela Faktów (FactSales):")
display(fact_sales.head())


Tabela Faktów (FactSales):


,Customer_SK,Product_SK,Date_SK,InvoiceNo,Quantity,Revenue
0,3797,3207,1,536365,6,15.30
1,3797,2619,1,536365,6,20.34
2,3797,2822,1,536365,8,22.00
3,3797,2771,1,536365,6,20.34
4,3797,2770,1,536365,6,20.34


## 7. SCD – DimCustomer

👉 Wskazówka:
- wybierz SCD1 lub SCD2
- jeśli SCD2 → dodaj kolumny:
  - valid_from
  - valid_to
  - current_flag

👉 Pomyśl:
- czy chcesz historię?


Dla wymiaru DimCustomer wybrałem implementację wolnozmiennych wymiarów typu 2 (SCD2 – wersjonowanie), aby śledzić i zachować historię zmian atrybutów klienta (np. przypisanego kraju). W tym celu do tabeli wymiaru wprowadziłem dodatkowe kolumny techniczne: datę od (valid_from), datę do (valid_to) oraz flagę aktualnego rekordu (current_flag). Konsekwencją tego podejścia jest zachowanie pełnej poprawności analiz historycznych – jeśli klient zmieni kraj zamieszkania, jego dawne transakcje nadal będą przypisane do państwa, w którym przebywał w momencie zakupu.

## 8. Wnioski

👉 Opisz:
- jakie decyzje podjąłeś
- czego się nauczyłeś
- jakie były trudności


W trakcie realizacji tego laboratorium podjąłem kluczowe decyzje projektowe, takie jak wybór najbardziej szczegółowego ziarna na poziomie pozycji faktury oraz zastosowanie kluczy sztucznych (surrogate keys), co zapewniło elastyczność analityczną i odporność modelu na zmiany w systemie źródłowym. Nauczyłem się, jak w praktyce transformować płaskie zbiory danych w relacyjny schemat gwiazdy przy użyciu biblioteki Pandas  oraz jak istotne dla jakości raportowania jest rygorystyczne wyczyszczenie danych przed ich załadowaniem. Największą trudność sprawiło mi zaimplementowanie mechanizmu wolnozmiennego wymiaru (SCD typu 2), a w szczególności poprawne wyliczenie dat obowiązywania historycznych rekordów (valid_from, valid_to) na podstawie statycznego pliku CSV.